# Week 1 Day 4 — Linear Regression From Scratch
**Jul 4, 2026**

Day 2 gave you `matmul`/`reshape`/broadcasting. Day 3 gave you autograd and `.backward()`. Today you combine them — but the point of "from scratch" is to derive and code the gradients **yourself**, by hand, with no `.backward()` call anywhere in the main training loop. Autograd only shows up at the end, as a way to check your own math.

This notebook is a scaffold, not a solution: every function below is a stub. Fill in each `# TODO`. Where useful there's a hint, but the formulas are for you to derive.

## Part 1: Setup — synthetic multi-feature dataset

Given, not an exercise: a factor-model-style dataset, `n_features` this time instead of just one — so your implementation has to work in matrix form, not just scalars like Day 3 Part 5.

In [1]:
import torch

torch.manual_seed(42)

n_samples, n_features = 200, 3

X = torch.randn(n_samples, n_features)
true_w = torch.tensor([1.5, -2.0, 0.5])
true_b = 0.3
noise = 0.1 * torch.randn(n_samples)

y = X @ true_w + true_b + noise   # shape (n_samples,)

print(X.shape, y.shape)

torch.Size([200, 3]) torch.Size([200])


## Part 2: Forward pass

TODO: implement `predict`. `w` is shape `(n_features,)`, `b` is a scalar tensor, `X` is `(n_samples, n_features)`. Output should be shape `(n_samples,)`.

Hint: this is a one-liner using `@` — think back to Day 2's broadcasting rules for how the bias scalar combines with a vector result.

In [3]:
def predict(X, w, b):
    # TODO: return the model's predictions, shape (n_samples,)
    return X @ w + b


# self-check: shape only, not correctness
w_test = torch.zeros(n_features)
b_test = torch.tensor(0.0)
out = predict(X, w_test, b_test)
assert out.shape == (n_samples,), f"expected shape ({n_samples},), got {tuple(out.shape)}"
print("shape OK")

shape OK


## Part 3: Loss — mean squared error

TODO: implement `mse_loss(y_pred, y)`. Should return a scalar tensor.

In [5]:
def mse_loss(y_pred, y):
    # TODO: mean squared error between y_pred and y
    MSE_LOSS = torch.mean((y_pred - y) ** 2)
    return MSE_LOSS

# self-check
loss_test = mse_loss(torch.tensor([1.0, 2.0]), torch.tensor([1.0, 4.0]))
assert loss_test.shape == (), "loss must be a scalar"
assert torch.isclose(loss_test, torch.tensor(2.0)), f"expected 2.0, got {loss_test.item()}"
print("loss fn OK")

loss fn OK


## Part 4: Manual gradients — the actual "from scratch" part

No `.backward()` here. Derive `dLoss/dw` and `dLoss/db` yourself using the chain rule:

- Start from `loss = mean((y_pred - y)^2)` and get `dLoss/dy_pred` first — that part is plain scalar calculus.
- Then use `y_pred = X @ w + b` to push that through to `w` and `b`. Think about shapes: `dLoss/dw` must come out as `(n_features,)`, and `dLoss/db` a scalar. That constraint tells you which `matmul`/transpose arrangement is the right one — there's only one that produces the correct shape.
- `b` is broadcast across all `n_samples` rows in the forward pass, so its gradient has to combine contributions from every row (think about what operation "undoes" a broadcast in the backward direction).

TODO: implement `compute_gradients`.

In [7]:
def compute_gradients(X, y, y_pred):
    # TODO: return (grad_w, grad_b) -- dLoss/dw shape (n_features,), dLoss/db scalar
    dloss_dy_pred = 2 * (y_pred - y) / y.shape[0]  # shape (n_samples,)
    grad_w = X.T @ dloss_dy_pred  # shape (n_features,)
    grad_b = torch.sum(dloss_dy_pred)  # scalar
    return grad_w, grad_b

### Check your derivation against autograd

Don't trust your hand-derived formula blindly — verify it. Run the same forward pass through autograd, `.backward()` it, and compare `w.grad`/`b.grad` against what your `compute_gradients` produces on the *same* `w`, `b`. If `torch.allclose` fails, the bug is in your derivation or your matmul/transpose arrangement, not in this check cell.

In [8]:
w_check = torch.tensor([0.2, -0.1, 0.4], requires_grad=True)
b_check = torch.tensor(0.1, requires_grad=True)

# autograd's version
y_pred_auto = predict(X, w_check, b_check)
loss_auto = mse_loss(y_pred_auto, y)
loss_auto.backward()

# your version, evaluated at the same w, b (detach so it's a plain tensor, no graph)
y_pred_manual = predict(X, w_check.detach(), b_check.detach())
grad_w_manual, grad_b_manual = compute_gradients(X, y, y_pred_manual)

print("autograd grad_w:", w_check.grad)
print("manual grad_w:  ", grad_w_manual)
print("autograd grad_b:", b_check.grad.item())
print("manual grad_b:  ", grad_b_manual.item() if torch.is_tensor(grad_b_manual) else grad_b_manual)

assert torch.allclose(w_check.grad, grad_w_manual, atol=1e-5), "grad_w mismatch — recheck your derivation"
assert torch.isclose(b_check.grad, torch.as_tensor(grad_b_manual), atol=1e-5), "grad_b mismatch — recheck your derivation"
print("\nmanual gradients match autograd")

autograd grad_w: tensor([-2.4672,  3.4364,  0.0046])
manual grad_w:   tensor([-2.4672,  3.4364,  0.0046])
autograd grad_b: -0.5782028436660767
manual grad_b:   -0.5782027244567871

manual gradients match autograd


## Part 5: Training loop

Now assemble it: forward pass → loss → your manual gradients → parameter update. No `.backward()`, no `torch.optim`, no `nn.Module` — every step should call functions you wrote above.

TODO: fill in the update step and the loss/gradient calls.

In [9]:
w = torch.zeros(n_features)
b = torch.tensor(0.0)

lr = 0.1
n_epochs = 100

for epoch in range(n_epochs):
    # TODO: forward pass -> y_pred
    y_pred = predict(X, w, b)
    # TODO: loss = mse_loss(...)
    loss = mse_loss(y_pred, y)
    # TODO: grad_w, grad_b = compute_gradients(...)
    grad_w, grad_b = compute_gradients(X, y, y_pred)
    # TODO: update w and b using grad_w, grad_b, and lr
    w -= lr * grad_w
    b -= lr * grad_b

    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch+1:3d} | loss: {loss.item():.4f} | w: {w.tolist()} | b: {b.item():.3f}")

print("\ntrue w:   ", true_w.tolist(), " true b:  ", true_b)
print("learned w:", w.tolist(), " learned b:", b.item())

epoch  20 | loss: 0.0129 | w: [1.4830735921859741, -1.9563769102096558, 0.4803336560726166] | b: 0.305
epoch  40 | loss: 0.0094 | w: [1.5070477724075317, -1.9956910610198975, 0.503515899181366] | b: 0.299
epoch  60 | loss: 0.0094 | w: [1.5075204372406006, -1.9965556859970093, 0.5043995976448059] | b: 0.298
epoch  80 | loss: 0.0094 | w: [1.5075312852859497, -1.9965767860412598, 0.5044298768043518] | b: 0.298
epoch 100 | loss: 0.0094 | w: [1.5075314044952393, -1.996577262878418, 0.5044307708740234] | b: 0.298

true w:    [1.5, -2.0, 0.5]  true b:   0.3
learned w: [1.5075314044952393, -1.996577262878418, 0.5044307708740234]  learned b: 0.2981604039669037


## Part 6 (bonus): Closed-form solution

You know this one from stats: OLS has a closed-form solution, no gradient descent required. If you augment `X` with a column of ones (to absorb the bias into the weight vector), the normal equation gives the exact least-squares solution in one shot.

TODO: implement `closed_form(X, y)` returning `(w, b)`. Useful PyTorch ops: concatenation for the ones-column, `.T` for transpose, `torch.linalg.inv` (or `torch.linalg.solve`, which avoids explicitly inverting).

Compare the result to what gradient descent converged to — they should be close.

In [10]:
def closed_form(X, y):
    # TODO: solve the normal equation for w and b
    y = y.view(-1, 1)  # reshape y to (n_samples, 1)
    X_b = torch.cat([X, torch.ones(X.shape[0], 1)], dim=1)  # add bias term
    w_b = torch.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y  # normal equation
    w_cf = w_b[:-1].view(-1)  # extract weights
    b_cf = w_b[-1].item()  # extract bias
    return w_cf, b_cf

w_cf, b_cf = closed_form(X, y)
print("closed-form w:", w_cf, " b:", b_cf)
print("gradient descent w:", w, " b:", b)

closed-form w: tensor([ 1.5075, -1.9966,  0.5044])  b: 0.2981607913970947
gradient descent w: tensor([ 1.5075, -1.9966,  0.5044])  b: tensor(0.2982)


## Try yourself

1. Add L2 (ridge) regularization: `loss = mse + alpha * (w ** 2).sum()`. Re-derive `grad_w` by hand — what extra term shows up?
2. Extend `closed_form` to solve *ridge* regression analytically (the regularized normal equation) and check it matches your regularized gradient descent.
3. Standardize `X` (zero mean, unit variance per column) before training — does gradient descent converge in fewer epochs at the same `lr`? Why would that be, given how `lr` interacts with feature scale?